# PaperAI — upload on your website, run OCR on Kaggle

This is a temporary authenticated API for testing the full `baidu/Unlimited-OCR` model
from your existing PaperAI website. **No paper dataset needs to be uploaded to Kaggle.**
The website sends each page through its Cloudflare Worker to this model, then formats
the returned text and exports Word as before.

Enable **GPU T4 x2** and **Internet**. Keep this notebook private. In **Add-ons → Secrets**,
create `UNLIMITED_OCR_API_KEY` with a private random key of at least 32 ASCII characters
and enable access for this notebook. Use the same value in the Cloudflare Worker's secret
of that name. Never publish the key in code, notebook output, GitHub or chat.

Run cells in order. The last cell starts a test connection for up to 30 minutes.
It prints a temporary API base URL. In Cloudflare Worker settings, set
`UNLIMITED_OCR_BASE_URL` to that URL (including `/v1`). The existing Worker adapter
will use Unlimited-OCR. If `OCR_PROVIDER` was explicitly set to `cloudflare-ai`, change
it to `unlimited-ocr`. Test one upload on the website and compare it with the scan.

Kaggle sessions and GPU quota are limited. The URL changes each time the tunnel starts.
This notebook does not provide permanent production hosting and does not bypass idle,
session or quota limits. Cloudflare Quick Tunnels are intended for testing only.


In [ ]:
# Preserve Kaggle's CUDA-matched torch/torchvision rather than replacing its GPU runtime.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.1", "Pillow==12.1.1", "einops==0.8.2",
    "addict==2.4.0", "easydict==1.13", "pymupdf==1.27.2.2", "psutil==7.2.2"])


In [ ]:
"""Full-weight Unlimited-OCR compatibility for GPUs without native BF16.

Only the inference methods' explicit image/autocast dtype is changed. The
model architecture, checkpoint and decoding algorithm remain the same.
"""
import ast
import inspect
import re
import textwrap
from types import MethodType


class _FP16Inference(ast.NodeTransformer):
    def __init__(self):
        self.replacements = 0

    def visit_Attribute(self, node):
        if isinstance(node.value, ast.Name) and node.value.id == "torch" and node.attr == "bfloat16":
            self.replacements += 1
            return ast.copy_location(ast.Attribute(value=node.value, attr="float16", ctx=node.ctx), node)
        return self.generic_visit(node)


def enable_fp16_inference(model):
    """Bind patched inference methods to this instance; never mutate torch."""
    patched = {}
    for name in ("infer", "infer_multi"):
        bound = getattr(model, name, None)
        if bound is None:
            continue
        original = inspect.unwrap(bound.__func__)
        if original.__code__.co_freevars:
            raise RuntimeError("Upstream inference changed: closures need compatibility review.")
        tree = ast.parse(textwrap.dedent(inspect.getsource(original)))
        functions = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == name]
        if len(functions) != 1:
            raise RuntimeError(f"Cannot safely locate upstream {name} method.")
        functions[0].decorator_list = []
        transformer = _FP16Inference()
        tree = transformer.visit(tree)
        ast.fix_missing_locations(tree)
        if not transformer.replacements:
            raise RuntimeError(f"Upstream {name} dtype changed; review before running on FP16 GPU.")
        namespace = dict(original.__globals__)
        exec(compile(tree, original.__code__.co_filename + ":kaggle-fp16", "exec"), namespace)
        patched[name] = (MethodType(namespace[name], model), transformer.replacements)
    if "infer" not in patched:
        raise RuntimeError("The checkpoint has no compatible infer method.")
    for name, (method, _) in patched.items():
        setattr(model, name, method)
    return {name: count for name, (_, count) in patched.items()}


def clean_transcription(raw):
    """Remove model grounding markup, preserving question text and numbering."""
    text = str(raw or "")
    text = re.sub(r"<\|ref\|>(?:text|title|table|image|equation|header|footer|caption|list)\s*<\|/ref\|>\s*(?=<\|det\|>)", "", text, flags=re.I)
    text = re.sub(r"<\|det\|>.*?<\|/det\|>", "", text, flags=re.S)
    text = re.sub(r"<\|/?ref\|>", "", text)
    text = re.sub(r"<\|(?:endoftext|eos|im_end|im_start)\|>|<｜end▁of▁sentence｜>", "", text)
    text = re.sub(r"^\s*```(?:markdown|md|text)?\s*\n(.*?)\n```\s*$", r"\1", text, flags=re.I | re.S)
    return text.strip()


In [ ]:
import gc, json, time, re
from pathlib import Path
import torch, torchvision, transformers
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import HfApi

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")
torch.cuda.set_device(0)
major, minor = torch.cuda.get_device_capability(0)
native_bf16 = major >= 8 and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if native_bf16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "compute capability:", (major, minor))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__,
      "transformers:", transformers.__version__, "dtype:", DTYPE)
print("Free GPU memory (GiB):", round(torch.cuda.mem_get_info(0)[0] / 1024**3, 2))

MODEL_ID = "baidu/Unlimited-OCR"
# Resolve and pin the complete official checkpoint/code to one revision for this run.
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
    trust_remote_code=True, use_safetensors=True, torch_dtype=DTYPE).eval().cuda()
patch_counts = {} if native_bf16 else enable_fp16_inference(model)
print("Loaded full model revision:", MODEL_REVISION, "FP16 inference adjustments:", patch_counts)


In [ ]:
"""Small authenticated API for temporary full-model website OCR sessions.

The predictor runs locally, serially. No uploaded images are saved as datasets.
This module has no model or tunnel side effects when imported.
"""
import base64
import binascii
import hmac
import json
import re
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

MODEL_ID = "baidu/Unlimited-OCR"
MAX_BODY = 12 * 1024 * 1024
MAX_IMAGE = 8 * 1024 * 1024


def parse_page(payload):
    if not isinstance(payload, dict) or payload.get("model") != MODEL_ID:
        raise ValueError("Select the complete baidu/Unlimited-OCR model.")
    if payload.get("stream", False):
        raise ValueError("This temporary API supports non-streaming requests.")
    messages = payload.get("messages")
    if not isinstance(messages, list) or len(messages) != 1 or messages[0].get("role") != "user":
        raise ValueError("Send one user message with one full-page image.")
    content = messages[0].get("content")
    if not isinstance(content, list) or len(content) != 2:
        raise ValueError("Send the document parsing prompt and one image.")
    prompts = [item.get("text") for item in content if isinstance(item, dict) and item.get("type") == "text"]
    images = [item.get("image_url", {}).get("url") for item in content if isinstance(item, dict) and item.get("type") == "image_url" and isinstance(item.get("image_url"), dict)]
    if prompts != ["<image>document parsing."] or len(images) != 1 or not isinstance(images[0], str):
        raise ValueError("Use the literal document parsing prompt and one data-URL image.")
    match = re.fullmatch(r"data:image/(jpeg|png|webp);base64,([A-Za-z0-9+/=]+)", images[0])
    if not match:
        raise ValueError("Only embedded JPEG, PNG or WebP images are supported.")
    try:
        data = base64.b64decode(match[2], validate=True)
    except (ValueError, binascii.Error):
        raise ValueError("The image has invalid base64 encoding.") from None
    if not data or len(data) > MAX_IMAGE:
        raise ValueError("Image must be nonempty and at most 8 MiB.")
    budget = payload.get("max_tokens", 8192)
    if isinstance(budget, bool) or not isinstance(budget, int) or not 1 <= budget <= 8192:
        raise ValueError("max_tokens must be an integer from 1 to 8192.")
    return data, match[1], budget


def create_server(predict, api_key, address=("127.0.0.1", 8000)):
    if not isinstance(api_key, str) or len(api_key) < 32 or not api_key.isascii() or api_key.strip() != api_key:
        raise ValueError("Set a private API key of at least 32 ASCII characters.")
    expected = ("Bearer " + api_key).encode("ascii")
    gpu_lock = threading.Lock()

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *_):
            pass  # Never print credentials or request data in notebook output.

        def reply(self, status, payload):
            body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
            self.send_response(status)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Cache-Control", "no-store")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            try:
                self.wfile.write(body)
            except (BrokenPipeError, ConnectionResetError):
                pass

        def authorized(self):
            received = self.headers.get("Authorization", "").encode("utf-8")
            if not hmac.compare_digest(received, expected):
                self.reply(401, {"error": "Authentication required."})
                return False
            return True

        def do_GET(self):
            if not self.authorized():
                return
            if self.path != "/health":
                return self.reply(404, {"error": "Not found."})
            self.reply(200, {"model": MODEL_ID, "temporary": True, "gpu_busy": gpu_lock.locked()})

        def do_POST(self):
            if not self.authorized():
                return
            if self.path != "/v1/chat/completions":
                return self.reply(404, {"error": "Not found."})
            self.connection.settimeout(30)
            try:
                length = int(self.headers.get("Content-Length", "0"))
                if not 0 < length <= MAX_BODY:
                    return self.reply(413, {"error": "Request body is empty or exceeds 12 MiB."})
                if self.headers.get("Content-Type", "").split(";")[0].strip() != "application/json":
                    return self.reply(415, {"error": "Send application/json."})
                data, mime, budget = parse_page(json.loads(self.rfile.read(length)))
            except (ValueError, TypeError, AttributeError, TimeoutError):
                return self.reply(400, {"error": "Invalid OCR request. Send one embedded page image."})
            if not gpu_lock.acquire(blocking=False):
                return self.reply(503, {"error": "GPU is processing another page. Retry after it finishes."})
            try:
                text, truncated = predict(data, mime, budget)
                if not isinstance(text, str) or not text.strip():
                    return self.reply(502, {"error": "The full model returned no transcription."})
                self.reply(200, {"model": MODEL_ID, "choices": [{"index": 0,
                    "message": {"role": "assistant", "content": text},
                    "finish_reason": "length" if truncated else "stop"}]})
            except ValueError:
                self.reply(400, {"error": "Invalid or oversized page image."})
            except Exception:
                self.reply(502, {"error": "Unlimited-OCR could not finish this page. Check the GPU session."})
            finally:
                gpu_lock.release()

    return ThreadingHTTPServer(address, Handler)


In [ ]:
import tempfile
from io import BytesIO
from PIL import Image

def predict_website_page(image_bytes, mime, requested_tokens):
    # Verify locally; never fetch an arbitrary remote image URL.
    with Image.open(BytesIO(image_bytes)) as probe:
        if probe.format not in {"JPEG", "PNG", "WEBP"} or probe.width * probe.height > 25000000:
            raise ValueError("Invalid image type or size.")
        probe.verify()
    generation = {}
    original_generate = model.generate
    had_instance_generate = "generate" in model.__dict__
    previous_generate = model.__dict__.get("generate")

    def budgeted_generate(*args, **kwargs):
        ids = kwargs.get("input_ids")
        if ids is None and args:
            ids = args[0]
        if ids is None:
            raise RuntimeError("Upstream generation interface changed.")
        input_length = int(ids.shape[-1])
        budget = min(requested_tokens, 32768 - input_length)
        if budget <= 0:
            raise ValueError("Page exceeds the model context window.")
        kwargs.pop("max_length", None)
        kwargs["max_new_tokens"] = budget
        output = original_generate(*args, **kwargs)
        sequences = output.sequences if hasattr(output, "sequences") else output
        generated = int(sequences.shape[-1]) - input_length
        eos = tokenizer.eos_token_id
        eos = eos if isinstance(eos, (list, tuple)) else [eos]
        generation["truncated"] = generated >= budget and int(sequences[0, -1]) not in eos
        return output

    model.generate = budgeted_generate
    try:
        with tempfile.TemporaryDirectory(prefix="paperai-page-") as directory:
            page = Path(directory) / ("page." + ("jpg" if mime == "jpeg" else mime))
            page.write_bytes(image_bytes)
            with torch.inference_mode():
                raw = model.infer(tokenizer, prompt="<image>document parsing.",
                    image_file=str(page), output_path=directory,
                    base_size=1024, image_size=640, crop_mode=True, max_length=32768,
                    no_repeat_ngram_size=35, ngram_window=128, temperature=0.0,
                    eval_mode=True, save_results=False)
            if not generation:
                raise RuntimeError("Generation did not return an output budget result.")
            return clean_transcription(raw), generation["truncated"]
    finally:
        if had_instance_generate:
            model.generate = previous_generate
        else:
            del model.generate
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
# This cell exposes an authenticated OCR test API, never a notebook/Jupyter service.
# It closes the server and tunnel after 30 minutes or when this cell is interrupted.
import hashlib, os, platform, subprocess, threading, urllib.request, time
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("UNLIMITED_OCR_API_KEY")
website_server = create_server(predict_website_page, api_key)
del api_key
website_thread = threading.Thread(target=website_server.serve_forever, daemon=True)
website_thread.start()
tunnel = None
try:
    # Download only the official Cloudflare release and verify its published SHA256.
    if platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("This tunnel setup expects Kaggle's Linux amd64 runtime.")
    request = urllib.request.Request("https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
                                     headers={"User-Agent": "PaperAI-Kaggle-Test"})
    with urllib.request.urlopen(request, timeout=30) as response:
        release = json.load(response)
    asset = next(item for item in release["assets"] if item["name"] == "cloudflared-linux-amd64")
    digest = asset.get("digest", "")
    if not re.fullmatch(r"sha256:[a-f0-9]{64}", digest):
        raise RuntimeError("The official release has no verifiable SHA256; stop and review the release.")
    asset_url = asset["browser_download_url"]
    if not asset_url.startswith("https://github.com/cloudflare/cloudflared/releases/download/"):
        raise RuntimeError("Unexpected Cloudflare release download origin.")
    with urllib.request.urlopen(asset_url, timeout=60) as response:
        binary = response.read(100 * 1024 * 1024 + 1)
    if len(binary) > 100 * 1024 * 1024 or hashlib.sha256(binary).hexdigest() != digest[7:]:
        raise RuntimeError("Cloudflare binary checksum verification failed.")
    binary_path = Path("/kaggle/working/cloudflared")
    binary_path.write_bytes(binary)
    binary_path.chmod(0o700)
    log_path = Path("/kaggle/working/paperai-tunnel.log")
    with log_path.open("w") as log:
        tunnel = subprocess.Popen([str(binary_path), "tunnel", "--url", "http://127.0.0.1:8000",
            "--protocol", "http2", "--no-autoupdate"], stdout=log, stderr=log)
    deadline = time.monotonic() + 60
    public_url = None
    while time.monotonic() < deadline:
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_path.read_text())
        if match:
            public_url = match[0]
            break
        if tunnel.poll() is not None:
            raise RuntimeError("Cloudflare tunnel stopped; check the tunnel log.")
        time.sleep(1)
    if not public_url:
        raise RuntimeError("No temporary tunnel URL appeared. Kaggle network access may block tunnels.")
    print("Temporary UNLIMITED_OCR_BASE_URL:", public_url + "/v1", flush=True)
    print("Set the matching API key privately in Cloudflare Worker secrets, then test a website upload.", flush=True)
    print("This test connection stops in 30 minutes. Interrupt this cell to stop sooner.", flush=True)
    # Serve only during this bounded test; no activity spoofing or quota bypass.
    stop_at = time.monotonic() + 30 * 60
    while time.monotonic() < stop_at:
        if tunnel.poll() is not None:
            raise RuntimeError("The tunnel stopped. Website OCR is no longer reachable.")
        time.sleep(1)
finally:
    if tunnel is not None:
        tunnel.terminate()
        try:
            tunnel.wait(timeout=10)
        except subprocess.TimeoutExpired:
            tunnel.kill()
            tunnel.wait(timeout=10)
    website_server.shutdown()
    website_server.server_close()
    print("Temporary website OCR connection stopped.", flush=True)


## After the test

Clear `UNLIMITED_OCR_BASE_URL` (and any explicit `OCR_PROVIDER=unlimited-ocr`) in Cloudflare
to restore the original provider after the Kaggle session ends. While Unlimited-OCR is
selected, an unavailable GPU returns an error; no smaller model is silently substituted.
Question numbering and continuous Word formatting are handled by the existing website.
Actual GPU latency and handwriting accuracy must be verified with a real website upload.
